In [13]:
import pathlib
import os 
import pandas as pd

train = "Dataset/Sunnybrook/Landmarks_2_10/train.txt"
val = "Dataset/Sunnybrook/Landmarks_2_10/val.txt"


train = pd.read_csv(train, header=None)
train = train.assign(split='train')
val = pd.read_csv(val, header=None)
val = val.assign(split='val')

train = pd.concat([train, val], ignore_index=True)
train = train.rename(columns={0: 'image_path'})
train

,image_path,split
0,SCD0000101_IM_0003_0108.png,train
1,SCD0000101_IM_0003_0079.png,train
2,SCD0000101_IM_0003_0068.png,train
3,SCD0000101_IM_0003_0099.png,train
4,SCD0000101_IM_0003_0179.png,train
...,...,...
722,SCD0004301_IM_0011_0080.png,val
723,SCD0004301_IM_0011_0147.png,val
724,SCD0004301_IM_0011_0087.png,val
725,SCD0004301_IM_0011_0020.png,val


In [14]:
image_base = "Dataset/Sunnybrook/Landmarks_2_10/images"
mask_base = "Dataset/Sunnybrook/Landmarks_2_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset125_Sunny/imagesTr"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset125_Sunny/labelsTr"

os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in train.iterrows():
    image_path = os.path.join(image_base, row[1]['image_path'])
    mask_path = os.path.join(mask_base, row[1]['image_path'])

    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    os.system("cp %s %s" % (image_path, image_dest))
    os.system("cp %s %s" % (mask_path, mask_dest))

    i += 1

train = train.assign(image_reference=image_reference)
train.to_csv("sunny_train_with_reference.csv", index=False)

In [15]:
test = "Dataset/Sunnybrook/Landmarks_2_10/test.txt"
test = pd.read_csv(test, header=None)
test

,0
0,SCD0000601_IM_0003_0140.png
1,SCD0000601_IM_0003_0067.png
2,SCD0000601_IM_0003_0060.png
3,SCD0000601_IM_0003_0127.png
4,SCD0000601_IM_0003_0147.png
...,...
80,SCD0003101_IM_0002_0166.png
81,SCD0003101_IM_0002_0106.png
82,SCD0003101_IM_0002_0020.png
83,SCD0003101_IM_0002_0080.png


In [16]:
image_base = "Dataset/Sunnybrook/Landmarks_2_10/images"
mask_base = "Dataset/Sunnybrook/Landmarks_2_10/masks"

i = 1

image_reference = []

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset125_Sunny/imagesTs"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset125_Sunny/labelsTs"

os.makedirs(nnUNet_image_base, exist_ok=True)
os.makedirs(nnUNet_mask_base, exist_ok=True)

for row in test.iterrows():
    image_path = os.path.join(image_base, row[1][0])
    mask_path = os.path.join(mask_base, row[1][0])

    image_new_name = "image_%s_0000.png" % i
    mask_new_name = "image_%s.png" % i

    image_reference.append("image_%s" % i)

    # copy image and mask
    image_dest = os.path.join(nnUNet_image_base, image_new_name)
    mask_dest = os.path.join(nnUNet_mask_base, mask_new_name)

    os.system("cp %s %s" % (image_path, image_dest))
    os.system("cp %s %s" % (mask_path, mask_dest))

    i += 1

test = test.assign(image_reference=image_reference)
test.to_csv("sunny_test_with_reference.csv", index=False)

In [18]:
import pandas as pd
import os

train = pd.read_csv("sunny_train_with_reference.csv")
val = train[train['split'] == 'val']
train = train[train['split'] == 'train']

folds = []
dictionary = {'train': train['image_reference'].tolist(),
              'val': val['image_reference'].tolist()}
folds.append(dictionary)

# save to json
import json
os.makedirs('/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_preprocessed/Dataset125_Sunny', exist_ok=True)
with open('/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_preprocessed/Dataset125_Sunny/splits_final.json', 'w') as f:
    json.dump(folds, f)

In [7]:
from medpy.metric.binary import dc, hd, __surface_distances
import numpy as np
import cv2 
import pandas as pd
import os

def calculate_metrics(pred, gt):
    dc_val = dc(pred, gt)
    hd_val = hd(pred, gt, voxelspacing=(1, 1))
    d1 = __surface_distances(pred, gt, voxelspacing=(1, 1))
    d2 = __surface_distances(gt, pred, voxelspacing=(1, 1))
    assd_val = np.mean(np.concatenate((d1, d2)))
    return dc_val, hd_val, assd_val

test = pd.read_csv("../sunny_test_with_reference.csv")

organs = [1,2,3]
organ_names = ['LV endo', 'LV epi']

nnUNet_image_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset125_Sunny/imagesTs"
nnUNet_mask_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset125_Sunny/labelsTs"

nnUNet_pred_base = "/media/ngaggion/SSDNVME/APOLO/nnUNet_files/nnUNet_raw/Dataset125_Sunny/imagesTs/Segmentation_FAST"

results = []

test['name'] = test["0"] 
test['label_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_mask_base, x + ".png"))
test['pred_path'] = test['image_reference'].apply(lambda x: os.path.join(nnUNet_pred_base, x + ".png"))

for row in test.iterrows():
    print(row[1]['name'])

    GT = cv2.imread(row[1]['label_path'], cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(row[1]['pred_path'].replace('.png', '_0000.png'), cv2.IMREAD_UNCHANGED)

    name = row[1]['name']

    for organ_num, organ_name in zip(organs, organ_names):
        pred = (mask == int(organ_num)).astype(np.uint8)
        gt = (GT == int(organ_num)).astype(np.uint8)
        if sum(gt.flatten()) == 0:
            continue
        elif sum(pred.flatten()) == 0:
            dc_val, hd_val, assd_val = 0, np.nan, np.nan
        else:
            dc_val, hd_val, assd_val = calculate_metrics(pred, gt)
        results.append({
            "image": name,
            "organ": organ_name,
            "dc": dc_val,
            "hd": hd_val,
            "assd": assd_val,
        })

results = pd.DataFrame(results)
results.to_csv("../Results/Sunnybrook/nnUNet_results2.csv", index=False)

SCD0000601_IM_0003_0140.png
SCD0000601_IM_0003_0067.png
SCD0000601_IM_0003_0060.png
SCD0000601_IM_0003_0127.png
SCD0000601_IM_0003_0147.png
SCD0000601_IM_0003_0080.png
SCD0000601_IM_0003_0160.png
SCD0000601_IM_0003_0040.png
SCD0000601_IM_0003_0120.png
SCD0000601_IM_0003_0020.png
SCD0000601_IM_0003_0047.png
SCD0000601_IM_0003_0100.png
SCD0000601_IM_0003_0027.png
SCD0000601_IM_0003_0007.png


/tmp/ipykernel_951058/4010960789.py:42: RuntimeWarning: overflow encountered in scalar add
  if sum(gt.flatten()) == 0:
/tmp/ipykernel_951058/4010960789.py:44: RuntimeWarning: overflow encountered in scalar add
  elif sum(pred.flatten()) == 0:


SCD0000601_IM_0003_0087.png
SCD0000601_IM_0003_0107.png
SCD0001801_IM_0003_0028.png
SCD0001801_IM_0003_0140.png
SCD0001801_IM_0003_0108.png
SCD0001801_IM_0003_0060.png
SCD0001801_IM_0003_0200.png
SCD0001801_IM_0003_0068.png
SCD0001801_IM_0003_0080.png
SCD0001801_IM_0003_0160.png
SCD0001801_IM_0003_0040.png
SCD0001801_IM_0003_0220.png
SCD0001801_IM_0003_0120.png
SCD0001801_IM_0003_0088.png
SCD0001801_IM_0003_0188.png
SCD0001801_IM_0003_0128.png
SCD0001801_IM_0003_0148.png
SCD0001801_IM_0003_0100.png
SCD0001801_IM_0003_0180.png
SCD0001801_IM_0003_0168.png
SCD0001801_IM_0003_0208.png
SCD0001801_IM_0003_0048.png
SCD0002001_IM_0010_0140.png
SCD0002001_IM_0010_0187.png
SCD0002001_IM_0010_0147.png
SCD0002001_IM_0010_0120.png
SCD0002001_IM_0010_0080.png
SCD0002001_IM_0010_0067.png
SCD0002001_IM_0010_0100.png
SCD0002001_IM_0010_0180.png
SCD0002001_IM_0010_0220.png
SCD0002001_IM_0010_0160.png
SCD0002001_IM_0010_0127.png
SCD0002001_IM_0010_0107.png
SCD0002001_IM_0010_0087.png
SCD0002001_IM_0010_0